# Statistical Analysis — LLM-Generated Code Vulnerability Study

Reads `data/master/master_dataset.csv` (produced by `merge_results.py`) and works through the study's four research questions:

1. How frequently do LLMs generate security vulnerabilities?
2. What vulnerability types are most common?
3. Does hybrid (static + dynamic) analysis detect more than static alone?
4. Does prompt style influence code security?

Run this after the full pipeline (`generate_code.py` → `static_analysis.py` → `dynamic_analysis.py` → `merge_results.py`) has produced a non-empty master dataset.

In [ ]:
import json
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")

df = pd.read_csv("data/master/master_dataset.csv")
print(f"{len(df)} programs loaded")
df.head()

## Data quality check

Before trusting any statistic below, check how many programs are missing a static or dynamic report — these were recorded as 0 vulnerabilities, which will bias rates downward if not accounted for.

In [ ]:
missing_static = (~df["has_static_report"]).sum()
missing_dynamic = (~df["has_dynamic_report"]).sum()
print(f"Missing static report: {missing_static} / {len(df)}")
print(f"Missing dynamic report: {missing_dynamic} / {len(df)}")

# Optional: drop incomplete rows for a stricter analysis.
# df = df[df["has_static_report"] & df["has_dynamic_report"]].copy()

## RQ1 — How frequently do LLMs generate vulnerabilities?

In [ ]:
df["is_vulnerable"] = df["total_vuln_count"] > 0

overall_rate = df["is_vulnerable"].mean()
print(f"Overall vulnerability rate: {overall_rate:.1%}  ({df['is_vulnerable'].sum()} / {len(df)} programs)")

rate_by_model = df.groupby("model")["is_vulnerable"].mean()
print("\nVulnerability rate by model:")
print(rate_by_model)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x="model", y="is_vulnerable", data=df, ax=ax, errorbar=("ci", 95))
ax.set_ylabel("Vulnerability rate")
ax.set_title("Vulnerability rate per model")
plt.tight_layout()
plt.show()

## RQ2 — What vulnerability types are most common?

`owasp_categories` is stored as a JSON string per row (`{category: count}`) — flatten it into a long-form table first.

In [ ]:
category_totals = Counter()
for raw in df["owasp_categories"]:
    try:
        counts = json.loads(raw)
    except (TypeError, json.JSONDecodeError):
        continue
    category_totals.update(counts)

category_df = (
    pd.DataFrame(category_totals.items(), columns=["category", "count"])
    .sort_values("count", ascending=False)
)
category_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x="count", y="category", data=category_df, ax=ax)
ax.set_title("Distribution of vulnerability categories (OWASP)")
plt.tight_layout()
plt.show()

## RQ3 — Does hybrid analysis detect more than static alone?

Paired comparison: for each program, static-only count vs. hybrid (static + dynamic) count. A paired test is appropriate here since both measurements come from the *same* program, not independent samples.

In [ ]:
static_only = df["static_vuln_count"]
hybrid = df["total_vuln_count"]

print(f"Mean static-only count: {static_only.mean():.2f}")
print(f"Mean hybrid count:      {hybrid.mean():.2f}")

# Paired t-test (assumes approx. normal differences)
t_stat, t_p = stats.ttest_rel(hybrid, static_only)
print(f"\nPaired t-test: t={t_stat:.3f}, p={t_p:.4g}")

# Wilcoxon signed-rank test (non-parametric alternative — safer given count data
# is unlikely to be normally distributed)
w_stat, w_p = stats.wilcoxon(hybrid, static_only)
print(f"Wilcoxon signed-rank test: W={w_stat:.3f}, p={w_p:.4g}")

In [ ]:
compare_df = pd.DataFrame({
    "Static only": static_only,
    "Hybrid (static + dynamic)": hybrid,
}).melt(var_name="Method", value_name="Vulnerability count")

fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(x="Method", y="Vulnerability count", data=compare_df, ax=ax)
ax.set_title("Static-only vs. hybrid detection")
plt.tight_layout()
plt.show()

## RQ4 — Does prompt style influence code security?

One-way ANOVA across the three prompt styles (Basic / Detailed / Minimal), using `total_vuln_count` as the outcome. Also reported per model, since prompt-style effects may not be uniform across models.

In [ ]:
groups = [g["total_vuln_count"].values for _, g in df.groupby("prompt_type")]
f_stat, anova_p = stats.f_oneway(*groups)
print(f"One-way ANOVA across prompt styles: F={f_stat:.3f}, p={anova_p:.4g}")

print("\nMean total vulnerabilities by prompt style:")
print(df.groupby("prompt_type")["total_vuln_count"].mean().sort_values())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x="prompt_type", y="total_vuln_count", hue="model", data=df, ax=ax, errorbar=("ci", 95))
ax.set_ylabel("Mean total vulnerabilities")
ax.set_title("Vulnerabilities per prompt type, split by model")
plt.tight_layout()
plt.show()

## Notes / next steps

- If the ANOVA is significant, follow up with a post-hoc test (e.g. Tukey HSD via `statsmodels`) to see *which* prompt-style pairs differ.
- Consider a two-way ANOVA (model × prompt_type) if you want to test for an interaction effect rather than looking at each factor separately.
- Revisit the data-quality check above before finalizing any numbers — programs with missing reports are currently treated as zero-vulnerability, which is optimistic.